<a href="https://colab.research.google.com/github/dtcec760-commits/quiz-6-3/blob/main/DuongThanhDanh_EEEEIU21025_Qu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np


def Sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

In [4]:
import numpy as np


def Softmax(x):
    x  = np.subtract(x, np.max(x))        # prevent overflow
    ex = np.exp(x)

    return ex / np.sum(ex)

In [7]:
import numpy as np

def MultiClass(W1, W2, X, D):
    alpha = 0.9

    N = 5
    for k in range(N):
        x = np.reshape(X[:,:,k], (25, 1))
        d = D[k, :].T

        v1 = np.matmul(W1, x)
        y1 = Sigmoid(v1)
        v  = np.matmul(W2, y1)
        y  = Softmax(v)

        e     = d - y
        delta = e

        e1     = np.matmul(W2.T, delta)
        delta1 = y1*(1-y1) * e1

        dW1 = alpha * delta1 * x.T
        W1  = W1 + dW1

        dW2 = alpha * delta * y1.T
        W2  = W2 + dW2

    return W1, W2

In [11]:
import os

os.makedirs("dataset/train/circle", exist_ok=True)
os.makedirs("dataset/train/star", exist_ok=True)
os.makedirs("dataset/train/pentagon", exist_ok=True)

os.makedirs("dataset/test/circle", exist_ok=True)
os.makedirs("dataset/test/star", exist_ok=True)
os.makedirs("dataset/test/pentagon", exist_ok=True)

print("Folders created successfully")

Folders created successfully


In [24]:
!rm -rf dataset/test/pentagon/.ipynb_checkpoints

In [42]:
!pip install imageio

In [48]:
import os
import numpy as np
import imageio.v2 as imageio

def load_images(folder):

    classes = ["circle","star","pentagon"]

    X = []
    D = []

    for label, shape in enumerate(classes):

        path = os.path.join(folder, shape)

        if not os.path.exists(path):
            continue

        for file in os.listdir(path):

            img_path = os.path.join(path, file)

            if not os.path.isfile(img_path):
                continue

            if not file.lower().endswith((".png",".jpg",".jpeg")):
                continue

            # read image
            img = imageio.imread(img_path)

            # convert to grayscale if needed
            if len(img.shape) == 3:
                img = np.mean(img, axis=2)

            # resize to 50x50
            from skimage.transform import resize
            img = resize(img,(50,50))

            img = img / 255.0
            img = img.reshape(2500,1)

            X.append(img)

            d = np.zeros((3,1))
            d[label] = 1
            D.append(d)

    return X, D


# -----------------------------
# Backpropagation
# -----------------------------
def MultiClass(W1,W2,X,D):

    alpha = 0.1

    for x,d in zip(X,D):

        # forward
        v1 = np.matmul(W1,x)
        y1 = Sigmoid(v1)

        v = np.matmul(W2,y1)
        y = Softmax(v)

        # error
        e = d - y

        # backprop
        delta2 = e
        delta1 = np.matmul(W2.T,delta2) * y1*(1-y1)

        # update
        W2 = W2 + alpha*np.matmul(delta2,y1.T)
        W1 = W1 + alpha*np.matmul(delta1,x.T)

    return W1,W2


# -----------------------------
# Main program
# -----------------------------
def TestMultiClass():

    # load training images
    X_train, D_train = load_images("dataset/train")

    # initialize weights
    W1 = 2*np.random.random((20,2500)) - 1
    W2 = 2*np.random.random((3,20)) - 1

    # training
    for epoch in range(10000):
        W1,W2 = MultiClass(W1,W2,X_train,D_train)

    print("Training finished")

    # testing
    X_test,_ = load_images("dataset/test")

    classes = ["Circle","Star","Pentagon"]

    for i,x in enumerate(X_test):

        v1 = np.matmul(W1,x)
        y1 = Sigmoid(v1)

        v = np.matmul(W2,y1)
        y = Softmax(v)

        print("Image",i+1)
        print("Prediction:",classes[np.argmax(y)])
        print("Circle probability:",y[0][0])
        print("Star probability:",y[1][0])
        print("Pentagon probability:",y[2][0])
        print()

    return W1,W2


if __name__ == "__main__":
    TestMultiClass()


Training finished
Image 1
Prediction: Circle
Circle probability: 0.9994052118063752
Star probability: 0.00010323377643643383
Pentagon probability: 0.0004915544171883246

Image 2
Prediction: Circle
Circle probability: 0.9994052285513141
Star probability: 0.0001031728457381217
Pentagon probability: 0.0004915986029478083

Image 3
Prediction: Star
Circle probability: 4.824680544191869e-05
Star probability: 0.9989734268413456
Pentagon probability: 0.0009783263532125088

Image 4
Prediction: Star
Circle probability: 0.005319076795337054
Star probability: 0.9942758238147742
Pentagon probability: 0.0004050993898886664

Image 5
Prediction: Pentagon
Circle probability: 0.00036171989106245335
Star probability: 0.0005203840593096677
Pentagon probability: 0.999117896049628

Image 6
Prediction: Pentagon
Circle probability: 0.0003500622451610707
Star probability: 0.0003947987258026886
Pentagon probability: 0.9992551390290362

